# Imports

In [137]:
import pandas as pd
from nb_utils import set_root
import numpy as np
import sys
import json
import os
from pathlib import Path
from typing import List, Union


PROJECT_DIR = set_root(2)

# Parameters

In [138]:
path_data = PROJECT_DIR / "data"
path_intermediate = path_data / "02_intermediate"
path_primary = path_data / "03_primary"

file_path_data = path_primary / "tracker_cut.parquet"
file_path_horm = path_intermediate / "data_horm_concat.parquet"
tracker_columns = ["tracker_id",	"class_id",	"x_min",	"y_min",	"x_max",	"y_max",	"x_center",	"y_center"]

# Data

In [139]:
data = pd.read_parquet(file_path_data)
data.head()
d = data[data['tracker_id'] ==6]
d[d['class_id'] == 367]
data.columns

Index(['tracker_id', 'class_id', 'x_min', 'y_min', 'x_max', 'y_max', 'ID',
       'x_center', 'y_center'],
      dtype='object')

# Functions

In [140]:
def set_root(level: int = 1) -> Path:
    for i in range(level):
        if i == 0:
            PROJECT_DIR = Path.cwd().parent
        else:
            PROJECT_DIR = PROJECT_DIR.parent
    sys.path.append(str(PROJECT_DIR))
    return PROJECT_DIR

def find_name_with_prefix(names: List[str], prefix: str) -> Union[None|str]:
    for name in names:
        if name.startswith(prefix):
            return name
    return None

def generate_path_url(content: Union[pd.DataFrame|np.ndarray], path_video: Union[Path|str], target_col: Union[str|int]= "ID"):
    path_url = {}
    for idx in content[target_col].unique():
        files_path = os.listdir(path_video)
        file_name = find_name_with_prefix(files_path, str(idx) + "_")
        if file_name:
            path_url[idx] = str(path_video / file_name)
    return path_url



def position_values(group):
    x = group['x_center'].iloc[-1]
    y = group['y_center'].iloc[-1]
    
    return pd.Series({'x': x, 'y': y})

def calculate_metrics(group):
    positions = list(zip(group['x_center'], group['y_center']))
    delta_t = 0.02
    
    vcl = calculate_vcl(positions, delta_t)
    vsl = calculate_vsl(positions, delta_t)
    vap = calculate_vap(positions, delta_t)
    alh = calculate_alh(positions)
    mad = calculate_mad(positions)
    last_positions = position_values(group)
    return pd.Series({'x': last_positions['x'], 'y': last_positions['y'] ,'VCL': vcl, 'VSL': vsl, 'VAP': vap, "ALH": alh, "MAD": mad})

def create_group_id(df, x):
    df['time'] = df.groupby('tracker_id').cumcount() // x
    #df['group_id'] = df['group_id'] + 1
    return df



#Funções Novas
def calculate_vcl(temp, delta_t=0.02):
    if len(temp) < 3:
        return 0
    a = []
    for idx in range(1, len(temp) - 1):
        x_center, y_center = temp[idx]
        x_center_minus, y_center_minus = temp[idx - 1]
        x_center_plus, y_center_plus = temp[idx + 1]
        norm_minus = np.linalg.norm(np.array([x_center, y_center]) - np.array([x_center_minus, y_center_minus]))
        norm_plus = np.linalg.norm(np.array([x_center_plus, y_center_plus]) - np.array([x_center, y_center]))
        numerador = norm_minus + norm_plus
        denominador = 2 * delta_t
        vci = numerador / denominador
        a.append(vci)
    return sum(a) / (len(temp) - 2)

def calculate_vsl(temp, delta_t):
    if len(temp) == 1:
        return 0
    init_value = temp[0]
    final_value = temp[-1]
    vsl = np.sqrt(((init_value[0] - final_value[0])**2) + ((init_value[1] - final_value[1]) ** 2))/(len(temp)*delta_t)
    return vsl

def calculate_vap(temp, delta_t=0.02):
    if len(temp) < 3:
        return 0
    a = []
    for idx in range(1, len(temp) - 1):
        x_center, y_center = temp[idx]
        x_center_minus, y_center_minus = temp[idx-1]
        a.append(np.sqrt(((x_center - x_center_minus)**(2)) + ((y_center - y_center_minus)**(2))))
    total_time = (len(temp) - 1) * delta_t
    return sum(a) / total_time
def calculate_alh(temp):
    mean_position = np.mean(temp, axis=0)
    a = []
    for idx in range(len(temp)):
        a.append(np.linalg.norm(mean_position - temp[idx]))
    alh = sum(a) / len(a)
    return alh

def calculate_mad(temp):
    if len(temp) < 3:
        return 0
    angles = []
    for idx in range(1, len(temp) - 1):
        p_i_minus_1 = np.array(temp[idx - 1])
        p_i = np.array(temp[idx])
        p_i_plus_1 = np.array(temp[idx + 1])

        vector_1 = p_i - p_i_minus_1
        vector_2 = p_i_plus_1 - p_i
        
        dot_product = np.dot(vector_1, vector_2)
        magnitude_1 = np.linalg.norm(vector_1)
        magnitude_2 = np.linalg.norm(vector_2)
        
        if magnitude_1 == 0 or magnitude_2 == 0:
            continue
        
        cos_theta = dot_product / (magnitude_1 * magnitude_2)
        theta = np.arccos(np.clip(cos_theta, -1.0, 1.0))
        angles.append(np.abs(theta))
    
    if len(angles) == 0:
        return 0
    
    mad = sum(angles) / len(angles)
    return mad


#Funções que geram os json
def metrics_dataframe_to_json(metrics_df, path_url_trackeado, path_url_nao_trackeado):
    data = {'metrics': []}

    for patient_id, patient_group in metrics_df.groupby('ID'):
        patient_metrics = {'id': int(patient_id), 'trackers': []}
        
        # Dicionário para armazenar tracker_ids por tipo de espermatozoide
        sperm_type_dict = {}

        for _, row in patient_group.iterrows():
            tracker_metrics = {
                'tracker_id': int(row['tracker_id']),
                'VCL': round(row['VCL'], 2),
                'VSL': round(row['VSL'], 2),
                'VAP': round(row['VAP'], 2),
                'ALH': round(row['ALH'], 2),
                'MAD': round(row['MAD'], 2),
                'type': row['tipo_espermatozoide'],
            }
            
            # Adicionar as métricas do tracker
            patient_metrics['trackers'].append(tracker_metrics)

            # Atualizar o dicionário de tipos de espermatozoides
            sperm_type = row['tipo_espermatozoide']
            if sperm_type not in sperm_type_dict:
                sperm_type_dict[sperm_type] = []
            sperm_type_dict[sperm_type].append(int(row['tracker_id']))

        # Adicionar URLs de vídeo
        patient_metrics["video_url_trackeado"] = path_url_trackeado.get(patient_id)
        patient_metrics["video_url_nao_trackeado"] = path_url_nao_trackeado.get(patient_id)

        # Adicionar o dicionário de tipos de espermatozoides
        patient_metrics["sperm_types"] = sperm_type_dict

        data['metrics'].append(patient_metrics)

    return json.dumps(data, indent=4)


def save_json_to_directory(json_data, filename):
    """
    Saves the JSON data to a specific directory 'visualization/outputs' located 
    one level up from the current working directory.

    Args:
        json_data (str): JSON formatted string.
        filename (str): Name of the JSON file.
    """
    parent_directory = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    target_directory = os.path.join(parent_directory, 'visualization', 'outputs')
    
    if not os.path.exists(target_directory):
        os.makedirs(target_directory)
    
    file_path = os.path.join(target_directory, filename)
    

    with open(file_path, 'w') as f:
        f.write(json_data)
    
    print(f"JSON saved to {file_path}")



def dataframe_to_json(df, grouped):
    """
    Converts the dataframe to the specified JSON format.

    Args:
        df (pd.DataFrame): DataFrame with metrics and positions.
        grouped (pd.DataFrame): Grouped DataFrame with additional route information.

    Returns:
        str: JSON formatted string.
    """
    data = {'individuos': []}
    grouped_dict = {}
    for _, row in grouped.iterrows():
        key = (row['ID'], row['tracker_id'])
        if key not in grouped_dict:
            grouped_dict[key] = []
        grouped_dict[key].append({'x': row['x_center'], 'y': row['y_center']})

    for patient_id, patient_group in df.groupby('ID'):
        individual = {'id': int(patient_id), 'espermatozoides': []}

        for tracker_id, group in patient_group.groupby('tracker_id'):
            espermatozoide = {'id': int(tracker_id), 'route': [], 'frames': []}
            key = (patient_id, tracker_id)
            if key in grouped_dict:
                espermatozoide['route'].extend(grouped_dict[key])

            for _, row in group.iterrows():
                espermatozoide['frames'].append({
                    'x': round(row['x'], 2), #ok
                    'y': round(row['y'], 2), #ok
                    'VCL': round(row['VCL'], 2),
                    'VSL': round(row['VSL'], 2),
                    'VAP': round(row['VAP'], 2),
                    'ALH': round(row['ALH'], 2),
                    'MAD': round(row['MAD'], 2)
                })

            individual['espermatozoides'].append(espermatozoide)
        data['individuos'].append(individual)

    return json.dumps(data, indent=4)


# Calculate metrics

In [141]:
data

,tracker_id,class_id,x_min,y_min,x_max,y_max,ID,x_center,y_center
0,0,0,81.517593,341.390228,98.150589,358.730377,6,89.834091,350.060303
1,1,0,220.177231,32.698105,235.510880,48.191841,6,227.844055,40.444973
2,2,0,501.025208,244.382629,518.151794,260.651978,6,509.588501,252.517303
3,3,0,168.232040,412.327515,191.797409,435.265625,6,180.014725,423.796570
4,4,0,445.648376,83.616653,466.867371,104.786850,6,456.257874,94.201752
...,...,...,...,...,...,...,...,...,...
2196284,2091,0,523.772156,161.258850,539.751770,181.233521,69,531.761963,171.246185
2196285,1829,0,461.298218,186.087952,476.646240,202.813721,69,468.972229,194.450836
2196286,3239,0,94.208496,424.543518,107.303009,438.854553,69,100.755753,431.699036
2196287,3254,0,435.511108,461.199219,446.554077,477.938171,69,441.032593,469.568695


In [142]:
#Calculo por janelas de tempo
fps = 50 #temos que as imagens foram captadas a uma velocidade de 50 frames por segundo
#grouped = data.sort_values(['ID', 'tracker_id']).drop('class_id', axis=1).reset_index()
#cria grupos de acordo com o fps que a gente definiu
grouped = data.groupby(['ID', 'tracker_id']).apply(create_group_id, x=fps).reset_index(drop=True)

#Calcula métricas de janela
result = grouped.groupby(['ID', 'tracker_id', 'time']).apply(calculate_metrics).reset_index()
result.head()

C:\Users\ccana\AppData\Local\Temp\ipykernel_35152\857757640.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = data.groupby(['ID', 'tracker_id']).apply(create_group_id, x=fps).reset_index(drop=True)
C:\Users\ccana\AppData\Local\Temp\ipykernel_35152\857757640.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = grouped.groupby(['ID', 'tracker_id', 'time']).apply(calculate_metrics).reset_in

,ID,tracker_id,time,x,y,VCL,VSL,VAP,ALH,MAD
0,1,0,0,152.466644,335.840515,0.0,0.0,0.0,0.0,0.0
1,1,1,0,492.019775,245.750488,0.0,0.0,0.0,0.0,0.0
2,1,2,0,270.667603,157.405945,0.0,0.0,0.0,0.0,0.0
3,1,3,0,340.559387,429.905579,0.0,0.0,0.0,0.0,0.0
4,1,4,0,635.276001,471.097137,0.0,0.0,0.0,0.0,0.0


In [143]:
result[result['ID']==6]

,ID,tracker_id,time,x,y,VCL,VSL,VAP,ALH,MAD


In [144]:
# Classificando os tipos de espermatozoides com base em porcentagens
def classify_sperm(row):
    # Tipo A: Rápidos e progressivos
    if(row['VCL'] >= row['VAP'] and row['VAP'] >= row['VSL']):

        if (row['VCL'] == 0 or row['VAP'] == 0 or row['VSL'] == 0) and row['MAD'] == 0:  # Imóveis
            return "Tipo D"
        
        elif row['VCL'] >= row['VSL'] * 8:  # Hiperativos: VCL 40% maior que VAP
            return "Tipo Hiperativo"
        
        elif ((round(row['VCL'] - row['VAP'], 0)  <= row['VSL'] * 0.1) # Diferença menor que 10% de VAP
            and (round(row['VAP'] - row['VSL'], 0)  <= row['VSL'] * 0.1) and row['VSL'] >= 40):  
            return "Tipo A"
        
        # Tipo B: Rápidos mas menos progressivos (VCL > VAP > VSL, mas com mais variação entre eles)
        elif (( round(row['VCL'] - row['VAP'], 0)  <= row['VSL'] * 0.1 ) and row['VSL'] > 20):
            return "Tipo B"
        
        # Tipo C: Movimentos fracos (VSL e/ou VAP baixos)
        elif (((row['VAP'] - row['VSL']) <= row['VSL'] * 0.5) ):  
            return "Tipo C"
        
        # Se não se encaixar em nenhuma das classificações acima
        else:
            return "Não Classificado" 

    else:
            return "Mal Identificado"





In [145]:
#Calculo de métricas gerais
metrics_df = data.groupby(['ID', 'tracker_id']).apply(calculate_metrics).reset_index()

# Aplicando a função para criar a nova coluna
metrics_df['tipo_espermatozoide'] = metrics_df.apply(classify_sperm, axis=1)
metrics_df = metrics_df[(metrics_df['tipo_espermatozoide'] != 'Não Classificado') & (metrics_df['tipo_espermatozoide'] != 'Mal Identificado')]

metrics_df.groupby(['tipo_espermatozoide']).size()



C:\Users\ccana\AppData\Local\Temp\ipykernel_35152\1394950486.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  metrics_df = data.groupby(['ID', 'tracker_id']).apply(calculate_metrics).reset_index()


tipo_espermatozoide
Tipo A               152
Tipo B             15292
Tipo C             14661
Tipo D             11193
Tipo Hiperativo     6706
dtype: int64

In [146]:
import pandas as pd

# Definindo os DataFrames dfm e dfr
dfm = pd.DataFrame({'ID': [1, 1, 1, 1, 2, 3, 4], 'tracker_id': [1, 2, 5, 6, 1, 2, 1], 'x': [1, 2, 5, 6, 1, 2, 1], 'y': [1, 2, 5, 6, 1, 2, 1]})
dfr = pd.DataFrame({'ID': [1, 1, 1, 1, 1, 1, 2, 3, 4], 'tracker_id': [1, 2, 2, 4, 5, 6, 1, 2, 1], 'ca': [1, 2, 2, 4, 5, 6, 1, 2, 1], 'sa': [1, 2, 2, 4, 5, 6, 1, 2, 1]})

# Criar um conjunto de combinações (ID, tracker_id) a partir de dfm
valid_combinations = set(zip(metrics_df ['ID'], metrics_df ['tracker_id']))

# Filtrar dfr para manter apenas as linhas cujas combinações (ID, tracker_id) estão em valid_combinations
filtered_result = result[result.apply(lambda row: (row['ID'], row['tracker_id']) in valid_combinations, axis=1)]

# Exibindo o DataFrame resultante
print(filtered_result)

       ID  tracker_id  time           x           y         VCL        VSL  \
0       1           0     0  152.466644  335.840515    0.000000   0.000000   
1       1           1     0  492.019775  245.750488    0.000000   0.000000   
2       1           2     0  270.667603  157.405945    0.000000   0.000000   
3       1           3     0  340.559387  429.905579    0.000000   0.000000   
4       1           4     0  635.276001  471.097137    0.000000   0.000000   
...    ..         ...   ...         ...         ...         ...        ...   
110716  9         656     0  401.296997   15.465927    0.000000   0.000000   
110719  9         659     0  504.468689  346.533508  137.401298  66.071188   
110722  9         662     0  568.947205  474.263214    0.000000   0.000000   
110727  9         667     0  569.693909  246.499084    0.000000   0.000000   
110728  9         668     0  394.786072   64.064178    0.000000   0.000000   

              VAP       ALH       MAD  
0        0.000000  0.00

In [147]:
filtered_result.columns

Index(['ID', 'tracker_id', 'time', 'x', 'y', 'VCL', 'VSL', 'VAP', 'ALH',
       'MAD'],
      dtype='object')

In [148]:
file_path_metrics_parquet = path_primary / "metrics2.parquet"
metrics_df.to_parquet(file_path_metrics_parquet, engine='pyarrow', index=False)

In [149]:
temp = metrics_df[metrics_df["ID"] == "47"].copy()
temp[temp["tracker_id"] == 19]

,ID,tracker_id,x,y,VCL,VSL,VAP,ALH,MAD,tipo_espermatozoide
36619,47,19,299.707947,60.25959,105.400111,49.913625,104.921995,75.864033,0.757528,Tipo B


In [150]:
d = metrics_df.groupby(['tipo_espermatozoide']).size().reset_index(name='count')
metrics_df.to_csv("metrics_df.csv");
d

,tipo_espermatozoide,count
0,Tipo A,152
1,Tipo B,15292
2,Tipo C,14661
3,Tipo D,11193
4,Tipo Hiperativo,6706


# Data to JSON

In [151]:
# URLs dos vídeos trackeados
path_video_trackeado = PROJECT_DIR / "data" / "03_primary" / "tracker_video"
path_url_trackeado = generate_path_url(metrics_df, path_video_trackeado)

# URLs dos vídeos não trackeados
path_video_nao_trackeado = PROJECT_DIR / "datasets" / "data" / "visem" / "visem-dataset" / "video_cut"
path_url_nao_trackeado = generate_path_url(metrics_df, path_video_nao_trackeado)

json_dt = metrics_dataframe_to_json(metrics_df, path_url_trackeado, path_url_nao_trackeado)
save_json_to_directory(json_dt, 'metrics_general2.json')

#json de acordo com a janela de tempo que foi definida no início
json_data = dataframe_to_json(filtered_result, grouped)
save_json_to_directory(json_data, 'data_window2.json')


JSON saved to c:\Users\ccana\Documents\Doutorado\VIS1170\cin-dataviz\visualization\outputs\metrics_general2.json
JSON saved to c:\Users\ccana\Documents\Doutorado\VIS1170\cin-dataviz\visualization\outputs\data_window2.json
